# Проверка min/max агрегированных TAG

Самостоятельный ноутбук: использует только библиотеки и `config.py`, которые уже применяются в `clustering_agg_v4.ipynb`. Никакие файлы не создаются.

In [ ]:
import pandas as pd

from config import (
    TAGS_EXCEL_PATH,
    DATA_CSV_PART1,
    DATA_CSV_PART2,
    FINAL_WEIGHTS,
    load_tag_lists,
)

## Загрузка данных — точно так же, как в итоговом ноутбуке

In [ ]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)

part1 = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter=',')
part2 = pd.read_csv(DATA_CSV_PART2, encoding='cp1251', delimiter=',')

casco_hashtags_full = pd.merge(
    part1,
    part2,
    on='POLICY_ZV',
    how='inner',
)
casco_hashtags_full.set_index('POLICY_ZV', inplace=True)

if 'TAG_JOIN_IND' in casco_hashtags_full.columns:
    casco_hashtags_full.drop(columns=['TAG_JOIN_IND'], inplace=True)

tag_columns = casco_hashtags_full.filter(like='TAG_').columns
casco_hashtags_full['SUM'] = (
    casco_hashtags_full[tag_columns]
    .fillna(0)
    .sum(axis=1)
)
casco_hashtags_full = casco_hashtags_full[
    casco_hashtags_full['SUM'] > 0
].copy()

print(f'Строк после объединения и фильтра SUM > 0: {len(casco_hashtags_full):,}')

## Расчёт raw score

In [ ]:
raw_scores = pd.DataFrame(index=casco_hashtags_full.index)

groups = {
    'auto_lover': 'auto_lover_list',
    'shopping': 'shopping_features_list',
}

for group_name, tag_list_key in groups.items():
    selected_tags = list(tag_lists[tag_list_key])
    weights = pd.Series(
        {
            tag: float(FINAL_WEIGHTS[group_name].get(tag, 0.0))
            for tag in selected_tags
        },
        dtype=float,
    )

    X_group = (
        casco_hashtags_full
        .reindex(columns=selected_tags, fill_value=0)
        .fillna(0)
        .astype(float)
    )

    raw_scores[f'{group_name}_raw_score'] = (
        X_group.mul(weights, axis=1).sum(axis=1)
    )

alcohol_tags = list(tag_lists['alcohol_features_list'])
X_alcohol = (
    casco_hashtags_full
    .reindex(columns=alcohol_tags, fill_value=0)
    .fillna(0)
    .astype(float)
)
raw_scores['alcohol_raw_score'] = X_alcohol.sum(axis=1)

## Min/max текущего датасета

In [ ]:
result = pd.DataFrame({
    'group': ['auto_lover', 'shopping', 'alcohol'],
    'raw_min': [
        raw_scores['auto_lover_raw_score'].min(),
        raw_scores['shopping_raw_score'].min(),
        raw_scores['alcohol_raw_score'].min(),
    ],
    'raw_max': [
        raw_scores['auto_lover_raw_score'].max(),
        raw_scores['shopping_raw_score'].max(),
        raw_scores['alcohol_raw_score'].max(),
    ],
    'theoretical_max': [
        sum(FINAL_WEIGHTS['auto_lover'].values()),
        sum(FINAL_WEIGHTS['shopping'].values()),
        len(alcohol_tags),
    ],
})

result

Сравните `raw_min` и `raw_max` после запуска на старом и новом датасете. Если для конкретной группы обе границы одинаковые, старый `MinMaxScaler.fit_transform()` использует одинаковую шкалу для этой группы.